<a href="https://colab.research.google.com/github/annkupina/ml_course_hw/blob/main/Anna_Kupina_HW_2_2_%D0%9D%D0%B5%D0%B7%D0%B1%D0%B0%D0%BB%D0%B0%D0%BD%D1%81%D0%BE%D0%B2%D0%B0%D0%BD%D0%B0_%D0%B1%D0%B0%D0%B3%D0%B0%D1%82%D0%BE%D0%BA%D0%BB%D0%B0%D1%81%D0%BE%D0%B2%D0%B0_%D0%BA%D0%BB%D0%B0%D1%81%D0%B8%D1%84%D1%96%D0%BA%D0%B0%D1%86%D1%96%D1%8F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

У цьому ДЗ ми потренуємось розв'язувати задачу багатокласової класифікації за допомогою логістичної регресії з використанням стратегій One-vs-Rest та One-vs-One, оцінити якість моделей та порівняти стратегії.

### Опис задачі і даних

**Контекст**

В цьому ДЗ ми працюємо з даними про сегментацію клієнтів.

Сегментація клієнтів – це практика поділу бази клієнтів на групи індивідів, які схожі між собою за певними критеріями, що мають значення для маркетингу, такими як вік, стать, інтереси та звички у витратах.

Компанії, які використовують сегментацію клієнтів, виходять з того, що кожен клієнт є унікальним і що їхні маркетингові зусилля будуть більш ефективними, якщо вони орієнтуватимуться на конкретні, менші групи зі зверненнями, які ці споживачі вважатимуть доречними та які спонукатимуть їх до купівлі. Компанії також сподіваються отримати глибше розуміння уподобань та потреб своїх клієнтів з метою виявлення того, що кожен сегмент цінує найбільше, щоб точніше адаптувати маркетингові матеріали до цього сегменту.

**Зміст**.

Автомобільна компанія планує вийти на нові ринки зі своїми існуючими продуктами (P1, P2, P3, P4 і P5). Після інтенсивного маркетингового дослідження вони дійшли висновку, що поведінка нового ринку схожа на їхній існуючий ринок.

На своєму існуючому ринку команда з продажу класифікувала всіх клієнтів на 4 сегменти (A, B, C, D). Потім вони здійснювали сегментовані звернення та комунікацію з різними сегментами клієнтів. Ця стратегія працювала для них надзвичайно добре. Вони планують використати ту саму стратегію на нових ринках і визначили 2627 нових потенційних клієнтів.

Ви маєте допомогти менеджеру передбачити правильну групу для нових клієнтів.

В цьому ДЗ використовуємо дані `customer_segmentation_train.csv`[скачати дані](https://drive.google.com/file/d/1VU1y2EwaHkVfr5RZ1U4MPWjeflAusK3w/view?usp=sharing). Це `train.csv`з цього [змагання](https://www.kaggle.com/datasets/abisheksudarshan/customer-segmentation/data?select=train.csv)

**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import f1_score, confusion_matrix, roc_curve, auc, root_mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

In [2]:
raw_df = pd.read_csv('customer_segmentation_train.csv')
raw_df

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
0,462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,D
1,462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4,A
2,466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,B
3,461735,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6,B
4,462669,Female,Yes,40,Yes,Entertainment,NaN,High,6.0,Cat_6,A
...,...,...,...,...,...,...,...,...,...,...,...
8063,464018,Male,No,22,No,NaN,0.0,Low,7.0,Cat_1,D
8064,464685,Male,No,35,No,Executive,3.0,Low,4.0,Cat_4,D
8065,465406,Female,No,33,Yes,Healthcare,1.0,Low,1.0,Cat_6,D
8066,467299,Female,No,27,Yes,Healthcare,1.0,Low,4.0,Cat_6,B


In [3]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8068 entries, 0 to 8067
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID               8068 non-null   int64  
 1   Gender           8068 non-null   object 
 2   Ever_Married     7928 non-null   object 
 3   Age              8068 non-null   int64  
 4   Graduated        7990 non-null   object 
 5   Profession       7944 non-null   object 
 6   Work_Experience  7239 non-null   float64
 7   Spending_Score   8068 non-null   object 
 8   Family_Size      7733 non-null   float64
 9   Var_1            7992 non-null   object 
 10  Segmentation     8068 non-null   object 
dtypes: float64(2), int64(2), object(7)
memory usage: 693.5+ KB


In [4]:
train_df, test_df = train_test_split(raw_df, test_size=0.2, random_state= 2, stratify= raw_df['Segmentation'])

In [5]:
input_cols = list(train_df.columns)[1:-1]
target_col = 'Segmentation'
train_inputs, train_targets = train_df[input_cols].copy(), train_df[target_col].copy()
test_inputs, test_targets = test_df[input_cols].copy(), test_df[target_col].copy()

In [6]:
numeric_cols = train_inputs.select_dtypes(include=np.number).columns.tolist()
categorical_cols = train_inputs.select_dtypes('object').columns.tolist()

imputer = SimpleImputer(strategy = 'mean').fit(train_inputs[numeric_cols])
train_inputs[numeric_cols] = imputer.transform(train_inputs[numeric_cols])
test_inputs[numeric_cols] = imputer.transform(test_inputs[numeric_cols])

imputer = SimpleImputer(strategy='constant', fill_value='missing').fit(train_inputs[categorical_cols])
train_inputs[categorical_cols] = imputer.transform(train_inputs[categorical_cols])
test_inputs[categorical_cols] = imputer.transform(test_inputs[categorical_cols])

scaler = MinMaxScaler().fit(train_inputs[numeric_cols])
train_inputs[numeric_cols] = scaler.transform(train_inputs[numeric_cols])
test_inputs[numeric_cols] = scaler.transform(test_inputs[numeric_cols])

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore').fit(train_inputs[categorical_cols])
encoded_cols = list(encoder.get_feature_names_out(categorical_cols))
train_inputs[encoded_cols] = encoder.transform(train_inputs[categorical_cols])
test_inputs[encoded_cols] = encoder.transform(test_inputs[categorical_cols])


**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

In [7]:
from imblearn.over_sampling import SMOTE, SMOTENC, SMOTEN
from imblearn.combine import SMOTETomek

In [8]:
#метод SMOTE
X_train_num = train_inputs[numeric_cols]
y_train = train_targets

smote = SMOTE(random_state=0)
X_train_smote, y_train_smote = smote.fit_resample(X_train_num, y_train)

In [9]:
#метод SMOTENC
train_inputs_for_smotenc = train_inputs[numeric_cols + categorical_cols]
cat_feature_indeces = [train_inputs_for_smotenc.columns.get_loc(c) for c in categorical_cols]

smotenc = SMOTENC(categorical_features=cat_feature_indeces, random_state=2)
X_train_smotenc, y_train_smotenc = smotenc.fit_resample(train_inputs_for_smotenc, train_targets)

In [10]:
#метод SMOTE-Tomek
smotetomek = SMOTETomek(random_state=0)
X_train_smotetomek, y_train_smotetomek = smotetomek.fit_resample(X_train_num, y_train)

In [11]:
#метод SMOTEN
X_train_cat = train_inputs[categorical_cols]

smoten = SMOTEN(random_state=2)
X_train_smoten, y_train_smoten= smoten.fit_resample(X_train_cat, y_train)

**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

In [12]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, precision_score, recall_score

In [13]:
X_train = train_inputs[numeric_cols + encoded_cols]
X_test = test_inputs[numeric_cols + encoded_cols]

#на оригінальних даних
log_reg = LogisticRegression(solver='liblinear')
ovr_model = OneVsRestClassifier(log_reg)
ovr_model.fit(X_train, y_train)
baseline_predictions = ovr_model.predict(X_test)

print(classification_report(test_targets, baseline_predictions))

              precision    recall  f1-score   support

           A       0.45      0.52      0.48       394
           B       0.35      0.15      0.21       372
           C       0.50      0.63      0.56       394
           D       0.65      0.71      0.68       454

    accuracy                           0.52      1614
   macro avg       0.49      0.50      0.48      1614
weighted avg       0.50      0.52      0.49      1614



In [14]:
#збалансованих з SMOTE даних
X_test = test_inputs[numeric_cols]

log_reg = LogisticRegression(solver='liblinear')
ovr_model = OneVsRestClassifier(log_reg)
ovr_model.fit(X_train_smote, y_train_smote)
smote_predictions = ovr_model.predict(X_test)

print(classification_report(test_targets, smote_predictions))

              precision    recall  f1-score   support

           A       0.32      0.31      0.31       394
           B       0.28      0.09      0.13       372
           C       0.37      0.46      0.41       394
           D       0.50      0.69      0.58       454

    accuracy                           0.40      1614
   macro avg       0.37      0.39      0.36      1614
weighted avg       0.37      0.40      0.37      1614



In [15]:
#збалансованих з SMOTE-Tomek даних
X_test = test_inputs[numeric_cols]

log_reg = LogisticRegression(solver='liblinear')
ovr_model = OneVsRestClassifier(log_reg)
ovr_model.fit(X_train_smotetomek, y_train_smotetomek)
smotetomek_predictions = ovr_model.predict(X_test)

print(classification_report(test_targets, smotetomek_predictions))

              precision    recall  f1-score   support

           A       0.32      0.30      0.31       394
           B       0.27      0.08      0.12       372
           C       0.37      0.47      0.42       394
           D       0.49      0.69      0.58       454

    accuracy                           0.40      1614
   macro avg       0.36      0.39      0.36      1614
weighted avg       0.37      0.40      0.37      1614



In [16]:
#збалансованих з SMOTENC даних
X_test = test_inputs[numeric_cols + encoded_cols]

X_train_smotenc_cat_encoded = encoder.transform(X_train_smotenc[categorical_cols])
X_train_smotenc_cat_encoded_df = pd.DataFrame(
    X_train_smotenc_cat_encoded,
    columns=encoded_cols,
    index=X_train_smotenc.index
)
X_train_smotenc_final = pd.concat(
    [X_train_smotenc[numeric_cols].reset_index(drop=True),
     X_train_smotenc_cat_encoded_df.reset_index(drop=True)],
    axis=1
)

log_reg = LogisticRegression(solver='liblinear')
ovr_model = OneVsRestClassifier(log_reg)
ovr_model.fit(X_train_smotenc_final, y_train_smotenc)
smotenc_predictions = ovr_model.predict(X_test)

print(classification_report(test_targets, smotenc_predictions))

              precision    recall  f1-score   support

           A       0.44      0.52      0.47       394
           B       0.35      0.21      0.26       372
           C       0.52      0.61      0.56       394
           D       0.66      0.66      0.66       454

    accuracy                           0.51      1614
   macro avg       0.49      0.50      0.49      1614
weighted avg       0.50      0.51      0.50      1614



In [17]:
#збалансованих з SMOTEN даних
X_test = test_inputs[encoded_cols]

X_train_smoten_encoded = encoder.transform(X_train_smoten)

log_reg = LogisticRegression(solver='liblinear')
ovr_model = OneVsRestClassifier(log_reg)
ovr_model.fit(X_train_smoten_encoded, y_train_smoten)
smoten_predictions = ovr_model.predict(X_test)

print(classification_report(test_targets, smoten_predictions))

              precision    recall  f1-score   support

           A       0.41      0.51      0.46       394
           B       0.33      0.21      0.26       372
           C       0.47      0.54      0.51       394
           D       0.66      0.64      0.65       454

    accuracy                           0.48      1614
   macro avg       0.47      0.47      0.47      1614
weighted avg       0.48      0.48      0.48      1614



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


In [18]:
results = {
    'baseline': round(f1_score(test_targets, baseline_predictions, average='macro'), 3),
    'SMOTE': round(f1_score(test_targets, smote_predictions, average='macro'), 3),
    'SMOTE-Tomek': round(f1_score(test_targets, smotetomek_predictions, average='macro'), 3),
    'SMOTENC': round(f1_score(test_targets, smotenc_predictions, average='macro'), 3),
    'SMOTEN': round(f1_score(test_targets, smoten_predictions, average='macro'), 3)
}
results

{'baseline': 0.483,
 'SMOTE': 0.359,
 'SMOTE-Tomek': 0.357,
 'SMOTENC': 0.488,
 'SMOTEN': 0.467}

Обрала метрику macro average для порівняння моделей. Найкращий результат показала модель на даних збалансованих з SMOTENC. Трохи ніжче за неї - модель на даних без ресемплінгу. Для цих моделей використовувались і числові і категоріальні ознаки.

Найгірші результати - для SMOTE і SMOTE-Tomek, які мають приблизно однаковий результат. Ці методи використовують тільки числові ознаки, можливо через це вони і показали найгірший результат. Бо моделі які тренувались на всіх ознаках показали себе краще.

Також модель на даних збалансованих з SMOTEN, тобто тільки на категоріальних ознаках, показала краще результат ніж попередні дві, то можемо зробити висновок, що категорільні ознаки є більш інформативними для цієї задачі ніж числові окремо, хоча найкращий результат отримуємо саме при використанні обох типів ознак разом.